In [1]:
from dotenv import load_dotenv

load_dotenv()

True

- web search를 할 수 있는 API 키가 필요하다

#### 기존 코드 불러오기

In [2]:
# 생성된 벡터스토어에 접근 -> 클래스만 바로 써야 함
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


embedding_function = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store = Chroma(
    embedding_function=embedding_function,
    collection_name='income_tax_collection',
    persist_directory='./income_tax_collection' 
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})  

In [3]:
# state 생성

from typing_extensions import TypedDict
from langchain_core.documents import Document
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    query: str  

    # 웹 서치와 벡터스토어의 문서 구조가 다르기 때문에 List로 선언 
    context:list # vector store 대상으로 검색을 했는데, 여기에서는 추가적으로 web search로 얻은 자료가 이 context에 들어간다
    answer: str


graph_builder = StateGraph(AgentState)  # 빌더 생성

In [4]:
# retrieve
def retrieve(state:AgentState):
    query = state['query']  
    docs = retriever.invoke(query)  
    return {'context' : docs}    

In [5]:
# llm과 prompt 분리 
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')  # 가격을 위해서 mini로 대체

In [6]:
from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser


generate_prompt = hub.pull("rlm/rag-prompt") 

# generate
def generate(state: AgentState):
    context = state['context']
    query = state['query']
    rag_chain = generate_prompt | llm | StrOutputParser()
    response = rag_chain.invoke({'question': query, 'context':context})
    return {'answer' : response} 

In [7]:
from langchain_classic import hub
from typing import Literal


doc_relevance_prompt = hub.pull("langchain-ai/rag-document-relevance")

def check_doc_relevance(state: AgentState) -> Literal['relevant', 'irrelevant']: 
    query = state['query']
    context = state['context']
    print(f'context == {context}') 
    doc_relevance_chain = doc_relevance_prompt | llm
    response = doc_relevance_chain.invoke({'question': query, 'documents':context})  
    print(f'doc relevance response == {response}') 

    if response['Score'] == 1:
        return 'relevant'
    
    return 'irrelevant'

#### 새로운 노드 생성

In [8]:
# rewrite

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 웹 검색에 용이하게 사용자의 질문 수정 -> 사전을 없애고, f-string 도 삭제해도 됨. 
rewrite_template = PromptTemplate.from_template("""
    사용자의 질문을 보고, 웹 검색에 용이하게 사용자의 질문을 수정해주세요
    질문 : {query}
    """
)

def rewrite(state: AgentState):
    query = state['query']
    rewrite_chain = rewrite_template | llm | StrOutputParser()  
    response = rewrite_chain.invoke({'query': query}) 
    
    print(f'rewrite question == {response}')  # 로그
    return {'query': response}


In [9]:
from langchain_tavily import TavilySearch
# from langchain_community.tools import TavilySearch   # cannot import 에러

tavily_search_tool = TavilySearch(
    max_results=3, 
    include_answer=True,
    include_raw_content=True,
    include_images=True,
    search_depth="advanced",
    
)

# tavily_search_tool.invoke({"query": "What happened at the last wimbledon"})

def web_search(state: AgentState):
    query = state['query']
    results = tavily_search_tool.invoke(query)  # List 형태 반환
    print(f'web_search question == {results}')  # 로그
    return {'context' : results}   # results가 List 형식


- vector store 과의 차이점? -> 구조가 다르다
    - vector store : 'metadata', 'page_content' 키
    - tavily : 'url', 'content' 키
- AgentState 는 Document 형식이 아닌 List로 선언 (문서 구조가 다르기 때문)

- rewrite를 활성화할 경우

In [ ]:
graph_builder.add_node('retrieve', retrieve)
graph_builder.add_node('generate', generate)
# graph_builder.add_node('check_doc_relevance', check_doc_relevance)  # 노드에 들어가면 안됨 -> 앞 강의 코드 참고
graph_builder.add_node('rewrite', rewrite)
graph_builder.add_node('web_search', web_search)

In [ ]:
# 엣지 추가

from langgraph.graph import START, END

graph_builder.add_edge(START, 'retrieve')
graph_builder.add_conditional_edges(
    'retrieve',
    check_doc_relevance,
    {
        'relevant' : 'generate',
        'irrelevant' : 'rewrite'
    }

)


graph_builder.add_edge('rewrite', 'web_search')
graph_builder.add_edge('web_search', 'generate')
graph_builder.add_edge('generate', END)

In [ ]:
from IPython.display import Image, display

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))  

#### initial_state

In [ ]:
initial_state = {'query' : '연봉 5천만원 거주자의 소득세는 얼마인가요?'}
graph.invoke(initial_state) # score == 1

In [ ]:
dummy_state = {'query' : '선유도역 맛집은?'}  # 엉뚱한 질문 넣어보기
graph.invoke(dummy_state)
# score == 0
# 답변을 웹 서치를 통해서 나옴

# query': '"선유도역 맛집 추천"' => 질문 수정한 것을 볼 수 있음

- 사용자의 질문을 수정하는 rewrite 과정이 필요한지?
    - 공손한 표현을 쓰면 더 잘 나온다는 말도 있지만, 크게 의미 X

In [ ]:
# rewrite 노드를 제거하고, 엣지 연결 수정

graph_builder.add_node('retrieve', retrieve)
graph_builder.add_node('generate', generate)
# graph_builder.add_node('rewrite', rewrite)
graph_builder.add_node('web_search', web_search)

In [ ]:
# 엣지 추가
from langgraph.graph import START, END

graph_builder.add_edge(START, 'retrieve')
graph_builder.add_conditional_edges(
    'retrieve',
    check_doc_relevance,
    {
        'relevant' : 'generate',
        'irrelevant' : 'web_search'  # rewrite 가 아니라 web_search로 바로 가도록
    }

)

# graph_builder.add_edge('rewrite', 'web_search')
graph_builder.add_edge('web_search', 'generate')
graph_builder.add_edge('generate', END)

In [ ]:
from IPython.display import Image, display

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))  

In [ ]:
dummy_state = {'query' : '선유도역 맛집은?'}  # 엉뚱한 질문 넣어보기
graph.invoke(dummy_state)
# score == 0
# 답변을 웹 서치를 통해서 나옴

# query': '"선유도역 맛집 추천"' => 질문 수정한 것을 볼 수 있음